1) Gerardus Cannavaro / 07
2) Razan Wira Aryanta / 20

## 3 Pertanyaan Awal

1. Produk dan kategori apa yang menghasilkan omzet tertinggi di kantin?

2. Seberapa besar peran pembeli borongan (jumlah > 10) terhadap total omzet, dan metode pembayaran apa yang paling sering dipakai?

3. Transaksi jumbo mana yang perlu dijaga stoknya, dan apakah ada input janggal (misal 500 pcs) yang merusak laporan?

In [1]:
import numpy as np
import pandas as pd

harga = np.array([5000, 7000, 3000, 12000, 4500])
print('Rata-rata harga:', harga.mean())
print('Harga tertinggi:', harga.max())
print('Harga setelah diskon 10%:', harga * 0.9)

Rata-rata harga: 6300.0
Harga tertinggi: 12000
Harga setelah diskon 10%: [ 4500.  6300.  2700. 10800.  4050.]


## Data Loading & Inspection


In [2]:
df = pd.read_csv('dataset_penjualan_kantin.csv')

print(df.head())  # 5 baris pertama
print(df.info())  # tipe data & jumlah non-null tiap kolom
print(df.describe(include='all'))  # statistik ringkas
print(df.shape)  # jumlah (baris, kolom)

  id_transaksi     tanggal  nama_produk kategori jumlah_terjual harga_satuan  \
0      TRX0042  2026-08-11   Roti Bakar  Makanan              9         7000   
1      TRX0005  2026-08-03     Gorengan  makanan              2         2000   
2      TRX0011  2026-08-04  Jus Alpukat  Minuman              4          NaN   
3      TRX0035  2026-08-10     Mie Ayam  Makanan            NaN        10000   
4      TRX0007  2026-08-03      Kerupuk    Snack             10      Rp2.000   

  nama_kasir metode_pembayaran  
0   Pak Agus             Tunai  
1     Bu Sri              QRIS  
2    Bu Wati             Tunai  
3   Pak Joko          Transfer  
4    Bu Wati          Transfer  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaksi       69 non-null     object
 1   tanggal            69 non-null     object
 2   nama_produk        69 non-

In [3]:
print(df.isnull().sum())  # jumlah data kosong tiap kolom
print('Duplikat ID:', df.duplicated(subset=['id_transaksi']).sum())
print(df[df.duplicated(subset=['id_transaksi'], keep=False)].sort_values('id_transaksi'))
print('\nKategori:', df['kategori'].unique())
print('\nTanggal contoh:', df['tanggal'].unique()[:10])
print('\nHarga contoh:', df['harga_satuan'].unique()[:10])
print('\nJumlah contoh:', df['jumlah_terjual'].unique()[:12])

id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64
Duplikat ID: 4
   id_transaksi         tanggal  nama_produk kategori jumlah_terjual  \
21      TRX0003  3 Agustus 2026  Nasi Goreng  MAKANAN              9   
51      TRX0003  3 Agustus 2026  Nasi Goreng  MAKANAN              9   
43      TRX0028  7 Agustus 2026     Mie Ayam  MAKANAN              4   
48      TRX0028  7 Agustus 2026     Mie Ayam  MAKANAN              4   
0       TRX0042      2026-08-11   Roti Bakar  Makanan              9   
61      TRX0042      2026-08-11   Roti Bakar  Makanan              9   
9       TRX0057      13/08/2026  Jus Alpukat  MINUMAN             14   
63      TRX0057      13/08/2026  Jus Alpukat  MINUMAN             14   

   harga_satuan nama_kasir metode_pembayaran  
21        12000   Pak Joko          Transfer  
51        12000   Pak Joko          Transfer 

## Cek Masalah Data

## Data Cleaning

In [4]:
df_clean = df.copy()

# 1. Duplikat ID -> hapus (1 ID = 1 struk)
print('Duplikat sebelum:', df_clean.duplicated(subset=['id_transaksi']).sum())
df_clean = df_clean.drop_duplicates(subset=['id_transaksi'])
print('Shape setelah drop duplikat:', df_clean.shape)

# 2. Kategori campur (Makanan/makanan/MAKANAN) -> samakan
df_clean['kategori'] = df_clean['kategori'].str.strip().str.title()
print(df_clean['kategori'].unique())

# 3. Jumlah: buang tulisan 'pcs', jadi angka
df_clean['jumlah_terjual'] = (df_clean['jumlah_terjual'].astype(str)
                              .str.lower()
                              .str.replace('pcs', '', regex=False)
                              .str.strip())
df_clean['jumlah_terjual'] = pd.to_numeric(df_clean['jumlah_terjual'], errors='coerce')

# 4. Buang outlier 500 pcs (tidak wajar, max normal 15)
df_clean = df_clean[~(df_clean['jumlah_terjual'] >= 100)]

# 5. Missing jumlah -> isi median (pakai NumPy seperti Latihan 1)
median_jumlah = int(np.median(df_clean['jumlah_terjual'].dropna()))
print('Median jumlah:', median_jumlah)
df_clean['jumlah_terjual'] = df_clean['jumlah_terjual'].fillna(median_jumlah).astype(int)

# 6. Harga: buang 'Rp' dan titik, jadi angka
df_clean['harga_satuan'] = (df_clean['harga_satuan'].astype(str)
                            .str.upper()
                            .str.replace('RP', '', regex=False)
                            .str.replace('.', '', regex=False)
                            .str.strip())
df_clean['harga_satuan'] = pd.to_numeric(df_clean['harga_satuan'], errors='coerce')
median_harga = int(np.median(df_clean['harga_satuan'].dropna()))
print('Median harga:', median_harga)
df_clean['harga_satuan'] = df_clean['harga_satuan'].fillna(median_harga).astype(int)

# 7. Tanggal multi-format -> ganti nama bulan jadi angka, lalu to_datetime
df_clean['tanggal'] = df_clean['tanggal'].astype(str).str.strip()
bulan = {'Januari': '01', 'Februari': '02', 'Maret': '03', 'April': '04',
         'Mei': '05', 'Juni': '06', 'Juli': '07', 'Agustus': '08',
         'September': '09', 'Oktober': '10', 'November': '11', 'Desember': '12'}
for nama, angka in bulan.items():
    df_clean['tanggal'] = df_clean['tanggal'].str.replace(nama, angka, case=False, regex=True)
df_clean['tanggal'] = pd.to_datetime(df_clean['tanggal'], dayfirst=True, errors='coerce')

# 8. Kasir kosong -> isi modus (paling sering muncul)
df_clean['nama_kasir'] = df_clean['nama_kasir'].fillna(df_clean['nama_kasir'].mode()[0])

print(df_clean.isnull().sum())  # harus 0 semua
print(df_clean.dtypes)

Duplikat sebelum: 4
Shape setelah drop duplikat: (65, 8)
['Makanan' 'Minuman' 'Snack']
Median jumlah: 7
Median harga: 8000
id_transaksi          0
tanggal              36
nama_produk           0
kategori              0
jumlah_terjual        0
harga_satuan          0
nama_kasir            0
metode_pembayaran     0
dtype: int64
id_transaksi                 object
tanggal              datetime64[ns]
nama_produk                  object
kategori                     object
jumlah_terjual                int64
harga_satuan                  int64
nama_kasir                   object
metode_pembayaran            object
dtype: object


## Data Manipulation (Filtering, Sorting, Groupby)


In [5]:
df_clean['total_pendapatan'] = df_clean['harga_satuan'] * df_clean['jumlah_terjual']  # kolom turunan

laris = df_clean[df_clean['jumlah_terjual'] > 10]  # filtering borongan
print('Transaksi borongan:', len(laris))
print(laris[['nama_produk', 'jumlah_terjual', 'total_pendapatan']])

urut = df_clean.sort_values(by='total_pendapatan', ascending=False)  # sorting
print(urut[['nama_produk', 'jumlah_terjual', 'total_pendapatan']].head())

ringkasan = df_clean.groupby('nama_produk')['total_pendapatan'].sum().sort_values(ascending=False)  # agregasi
print(ringkasan)
print(df_clean.groupby('kategori')['total_pendapatan'].sum())

Transaksi borongan: 15
         nama_produk  jumlah_terjual  total_pendapatan
6          Teh Botol              14             70000
7           Mie Ayam              14            140000
9        Jus Alpukat              14            112000
12         Teh Botol              12             60000
14  Keripik Singkong              15             45000
15       Nasi Goreng              13            156000
16       Jus Alpukat              15            120000
20       Nasi Goreng              15            180000
26             Bakso              11            121000
29         Nasi Uduk              14            112000
30          Mie Ayam              13            130000
32       Nasi Goreng              12            144000
44       Nasi Goreng              11            132000
46       Jus Alpukat              13            104000
57            Es Teh              14             42000
    nama_produk  jumlah_terjual  total_pendapatan
20  Nasi Goreng              15            1800

## Simpan + Profiling Summary

In [6]:
df_clean.to_csv('dataset_bersih.csv', index=False)
print(df_clean.shape)
print('File dataset_bersih.csv tersimpan')

(64, 9)
File dataset_bersih.csv tersimpan


**Temuan 1: Makanan raja omzet, Jus Alpukat raja satuan.** Groupby menunjukkan Makanan (Nasi Goreng 10x, Mie Ayam 10x, Rp8-12rb) total omzetnya tertinggi. Tapi per produk, Jus Alpukat (11 transaksi) paling sering keluar. Kesimpulan: Makanan untuk profit, Minuman untuk traffic.

**Temuan 2: Borongan (qty 11-15) penopang omzet.** Filtering qty>10 dan sorting total_pendapatan menunjukkan struk terbesar selalu borongan (misal Nasi Goreng 15x = Rp180rb). Outlier 500 pcs terbukti salah input dan sudah dibuang supaya tidak jadi juara palsu.

**Temuan 3: Data awal kotor, cleaning simpel ala modul cukup.** 4 ID duplikat di-drop, kategori di-title-case, Rp/pcs dibuang, missing di-fill median/modus, tanggal bulan-Indonesia diganti angka. Hasil akhir 0 missing, tipe sudah int/datetime, siap divisualisasikan.

## Lembar Refleksi

### 6. Tahap mana (loading, inspection, cleaning, atau manipulation) yang paling menantang bagi kelompok kalian, dan mengapa?
Tahap **cleaning** paling menantang karena tanggal 35 beda format (kata Agustus gagal di-parse), jadi kami ganti nama bulan jadi angka dulu.

### 7. Menurutmu, mengapa keputusan membersihkan data (mis. menghapus vs mengisi data kosong) perlu didasarkan pada alasan yang jelas, bukan asal-asalan?
Karena tiap pilihan dapat mengubah omzet laporan. Contoh nyata di dataset ini: harga kosong TRX0011 (Jus Alpukat) jika di-drop menghilangkan 1 transaksi, jika diisi 0 omzetnya Rp0 (salah), jika diisi median Jus Alpukat Rp8000 hasilnya realistis. TRX0050 500 pcs jika dibiarkan, Roti Bakar langsung jadi juara omzet palsu Rp3,5 juta dan mengalahkan Nasi Goreng/Mie Ayam. Pemilik kantin bisa salah stok dan salah hitung keuntungan atau kerugian.

### 8. Apa hubungan antara dataset bersih hasil proyek ini dengan pekerjaan seorang Data Analyst di dunia nyata?
Data Analyst tidak pernah dapat data kasir yang rapi. Di dunia nyata struk juga double-input (ID ganda), kasir beda gaya ketik (MAKANAN vs Makanan), format tanggal beda mesin, harga ke-copy dengan Rp, dan salah ketik 500 vs 5. Tugas analyst adalah mengubah 69 baris kotor menjadi `dataset_bersih.csv` 65 transaksi unik yang tipenya benar sebelum divisualisasikan di Elemen 3. Grafik dari data kotor akan menyesatkan (misal tanggal tidak bisa diurutkan tren harian), sedangkan dari data bersih bisa dipakai untuk keputusan stok borongan dan promo.